# LendingClub Credit Risk & Loan Default Prediction: Exploratory Data Analysis (EDA)

## Objectives
1. **Cohort Analysis & Target Definition**: Investigate empirical default distribution ($y=0$ vs $y=1$) post-hygiene filtering of active/ongoing loans.
2. **Risk Drivers by Grade & Sub-Grade**: Assess how default likelihood scales across credit risk tiers.
3. **Domain Financial Features**: Analyze engineered ratios (`installment_to_income`, `credit_history_length`, `unutilized_credit_limit`, `revolving_debt_to_income`).
4. **Weight of Evidence (WoE) & Information Value (IV)**: Establish predictive separation power for Basel scorecards.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

# Ensure src is importable
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

from src.feature_engineering import CreditFeatureEngineer, calculate_woe_iv

### 1. Load Processed Dataset

In [ ]:
data_path = ROOT_DIR / "data" / "processed" / "cleaned_loans.parquet"
if not data_path.exists():
    data_path = ROOT_DIR / "data" / "processed" / "cleaned_loans.csv"

df = pd.read_parquet(data_path) if data_path.suffix == ".parquet" else pd.read_csv(data_path)
print(f"Loaded dataset shape: {df.shape}")
df.head()

### 2. Default Target Distribution ($y$)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

target_counts = df["default_flag"].value_counts()
labels = ["Fully Paid (0)", "Default / Charged Off (1)"]
colors = ["#2ecc71", "#e74c3c"]

ax[0].pie(target_counts, labels=labels, autopct="%1.1f%%", colors=colors, startangle=140, explode=(0, 0.08))
ax[0].set_title("Portfolio Default Distribution", fontsize=13, fontweight="bold")

sns.barplot(x=labels, y=target_counts.values, palette=colors, ax=ax[1])
ax[1].set_ylabel("Loan Volume")
ax[1].set_title("Loan Count by Outcome", fontsize=13, fontweight="bold")
for p in ax[1].patches:
    ax[1].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                   ha='center', va='center', color='white', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

### 3. Empirical Default Rate by Grade & Sub-Grade

In [ ]:
grade_perf = df.groupby("grade")["default_flag"].agg(total="count", defaults="sum", rate=lambda x: np.mean(x) * 100)
display(grade_perf)

plt.figure(figsize=(10, 5))
sns.barplot(x=grade_perf.index, y=grade_perf["rate"], palette="Spectral_r")
plt.title("Empirical Default Rate (%) across Basel Risk Grades", fontsize=14, fontweight="bold")
plt.ylabel("Default Rate (%)")
plt.xlabel("LendingClub Credit Grade")
plt.ylim(0, 60)
for i, v in enumerate(grade_perf["rate"]):
    plt.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontweight="bold")
plt.show()

### 4. Feature Engineering: Domain Financial Features

In [ ]:
engineer = CreditFeatureEngineer(cap_percentile=0.99)
engineer.fit(df.drop(columns=["default_flag"]), df["default_flag"])
df_features = engineer.transform(df.drop(columns=["default_flag"]))
df_features["default_flag"] = df["default_flag"].values

ratio_cols = ["installment_to_income", "credit_history_length", "unutilized_credit_limit", "revolving_debt_to_income"]
df_features[ratio_cols].describe()

### 5. Weight of Evidence (WoE) and Information Value (IV) Analysis

In [ ]:
iv_data = pd.DataFrame(list(engineer.iv_summary_.items()), columns=["Feature", "Information_Value"])
iv_data = iv_data.sort_values(by="Information_Value", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=iv_data, x="Information_Value", y="Feature", palette="viridis")
plt.title("Information Value (IV) of Core Credit Predictors", fontsize=14, fontweight="bold")
plt.axvline(0.3, color="red", linestyle="--", label="Strong Predictor Threshold (IV >= 0.3)")
plt.legend()
plt.show()